In [11]:
!pip install streamlit
!pip install pyngrok


In [13]:
# ========================================
# CELL 1: INSTALL PACKAGES FOR STREAMLIT
# ========================================

# Install required packages
!pip install streamlit plotly wordcloud scikit-learn textblob pyngrok -q

# Install ngrok for tunneling (to access Streamlit from Colab)
!wget -q -c -nc https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip -qq -n ngrok-stable-linux-amd64.zip

print("✅ All packages installed successfully!")


✅ All packages installed successfully!


In [47]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
from textblob import TextBlob
import warnings
warnings.filterwarnings('ignore')

# Machine Learning imports
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import os

# Page configuration
st.set_page_config(
    page_title="🚨 Disaster Response Analytics",
    page_icon="🚨",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS
st.markdown("""
<style>
    .main-header {
        font-size: 3rem;
        color: #FF4B4B;
        text-align: center;
        margin-bottom: 2rem;
        text-shadow: 2px 2px 4px rgba(0,0,0,0.3);
    }
    .metric-card {
        background: linear-gradient(90deg, #FF6B6B, #4ECDC4);
        padding: 1rem;
        border-radius: 10px;
        color: white;
        text-align: center;
        margin: 0.5rem 0;
    }
    .prediction-result {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 1.5rem;
        border-radius: 15px;
        color: white;
        margin: 1rem 0;
        box-shadow: 0 4px 15px rgba(0,0,0,0.2);
    }
    .sidebar .sidebar-content {
        background: linear-gradient(180deg, #2196F3, #21CBF3);
    }
</style>
""", unsafe_allow_html=True)

# =======================
# Fungsi caching (di luar class)
# =======================
@st.cache_data
def load_data(file_path):
    """Memuat data pesan bencana"""
    try:
        df = pd.read_csv(file_path)
        return df
    except Exception as e:
        st.error(f"Error loading data: {e}")
        return None

@st.cache_resource
def train_models(df):
    """Melatih model ML untuk klasifikasi tipe bencana"""
    X = df['message'].astype(str)
    y = df['disaster_type']

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )

    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    models = {
        'Naive Bayes': MultinomialNB(),
        'Logistic Regression': LogisticRegression(random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
    }

    results = {}
    trained_models = {}

    for name, model in models.items():
        model.fit(X_train_vec, y_train)
        y_pred = model.predict(X_test_vec)
        accuracy = accuracy_score(y_test, y_pred)

        trained_models[name] = model
        results[name] = accuracy

    return trained_models, vectorizer, label_encoder, results

# =======================
# Class Dashboard
# =======================
class DisasterAnalyticsDashboard:
    def __init__(self):
        self.df = None

    def extract_disaster_type(self, message):
        if pd.isna(message):
            return 'unknown'

        message_lower = message.lower()
        disaster_keywords = {
            'earthquake': ['earthquake', 'quake', 'seismic', 'tremor'],
            'flood': ['flood', 'flooding', 'water', 'river', 'overflow'],
            'hurricane': ['hurricane', 'storm', 'cyclone', 'wind'],
            'fire': ['fire', 'burning', 'flame', 'smoke'],
            'medical': ['medical', 'health', 'hospital', 'doctor', 'sick'],
            'food': ['food', 'hungry', 'eat', 'water', 'drink'],
            'shelter': ['shelter', 'house', 'home', 'roof', 'building'],
            'rescue': ['rescue', 'help', 'trapped', 'stuck', 'save'],
            'infrastructure': ['road', 'bridge', 'power', 'electricity', 'communication']
        }

        for disaster_type, keywords in disaster_keywords.items():
            if any(keyword in message_lower for keyword in keywords):
                return disaster_type

        return 'general'

    def get_sentiment(self, message):
        if pd.isna(message):
            return 'neutral'

        try:
            blob = TextBlob(message)
            polarity = blob.sentiment.polarity

            if polarity > 0.1:
                return 'positive'
            elif polarity < -0.1:
                return 'negative'
            else:
                return 'neutral'
        except:
            return 'neutral'

    def extract_urgency_level(self, message):
        if pd.isna(message):
            return 'low'

        message_lower = message.lower()
        urgent_keywords = ['urgent', 'emergency', 'critical', 'immediate', 'asap', 'help']
        high_keywords = ['important', 'serious', 'severe', 'major']

        if any(keyword in message_lower for keyword in urgent_keywords):
            return 'urgent'
        elif any(keyword in message_lower for keyword in high_keywords):
            return 'high'
        else:
            return 'medium'

    def preprocess_data(self, df):
        df = df.dropna()
        df['disaster_type'] = df['message'].apply(self.extract_disaster_type)
        df['sentiment'] = df['message'].apply(self.get_sentiment)
        df['message_length'] = df['message'].str.len()
        df['urgency_level'] = df['message'].apply(self.extract_urgency_level)
        return df

# =======================
# Fungsi Halaman (placeholder)
# =======================
def show_overview(df):
    st.subheader("📊 Overview Dataset")
    st.write(df.head())
    st.write(f"Jumlah data: {len(df)}")

def show_data_analysis(df):
    st.subheader("📈 Analisis Data")
    st.bar_chart(df['disaster_type'].value_counts())

def show_prediction_page(df, dashboard):
    st.subheader("🔮 Prediksi AI")
    models, vectorizer, encoder, results = train_models(df)
    st.write("Akurasi model:")
    st.write(results)

def show_risk_assessment(df):
    st.subheader("📉 Penilaian Risiko")
    st.write("Coming soon...")

def show_data_upload_page(df):
    st.subheader("📁 Data yang Diunggah")
    st.write(df)

# =======================
# Main App
# =======================
def main():
    st.markdown('<h1 class="main-header">🚨 Disaster Response Analytics Dashboard</h1>',
                unsafe_allow_html=True)

    dashboard = DisasterAnalyticsDashboard()

    with st.sidebar:
        st.header("📋 Navigasi")
        page = st.selectbox(
            "Pilih halaman:",
            ["🏠 Overview", "📊 Data Analysis", "🔮 Prediksi AI", "📈 Risk Assessment", "💾 Unggah Data"]
        )

        st.markdown("---")
        st.markdown("### 📁 Info Dataset")

        uploaded_file = st.file_uploader("Unggah file CSV", type=['csv'])

        if uploaded_file is None:
            file_path = "/Users/rullbachtiar/Downloads/Folder Baru Dengan Item/disaster_messages.csv"
            if os.path.exists(file_path):
                st.success("✅ Menggunakan dataset default")
                data = load_data(file_path)
            else:
                st.warning("⚠️ Dataset default tidak ditemukan. Silakan unggah file CSV.")
                data = None
        else:
            with open("temp_disaster_data.csv", "wb") as f:
                f.write(uploaded_file.getbuffer())
            data = load_data("temp_disaster_data.csv")
            st.success("✅ File berhasil diunggah!")

    if data is not None:
        processed_data = dashboard.preprocess_data(data.copy())

        if page == "🏠 Overview":
            show_overview(processed_data)
        elif page == "📊 Data Analysis":
            show_data_analysis(processed_data)
        elif page == "🔮 Prediksi AI":
            show_prediction_page(processed_data, dashboard)
        elif page == "📈 Risk Assessment":
            show_risk_assessment(processed_data)
        elif page == "💾 Unggah Data":
            show_data_upload_page(processed_data)
    else:
        st.error("❌ Tidak ada data. Silakan unggah file CSV atau periksa path default.")

if __name__ == "__main__":
    main()


2025-06-27 05:47:07.266 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-27 05:47:07.270 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-27 05:47:07.271 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-27 05:47:07.272 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-27 05:47:07.274 No runtime found, using MemoryCacheStorageManager
2025-06-27 05:47:07.281 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-27 05:47:07.283 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-27 05:47:07.284 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-27 05:47:07.285 Thread 'MainThread':

In [23]:
# ========================================
# CELL 3: SETUP NGROK FOR COLAB ACCESS
# ========================================

# Set up ngrok authentication (you need to get your token from ngrok.com)
import getpass

print("🔐 Ngrok Setup")
print("1. Go to https://ngrok.com/")
print("2. Sign up for free account")
print("3. Get your authtoken from dashboard")
print("4. Enter it below:")

# Uncomment the line below and enter your ngrok token
ngrok_token = getpass.getpass("2z4dv70d99S6fD3J4dkvZreO2up_2SC35TqZjhDUgzc7hhfPm")
!./ngrok authtoken {ngrok_token}

print("⚠️ Please uncomment the lines above and add your ngrok token!")


🔐 Ngrok Setup
1. Go to https://ngrok.com/
2. Sign up for free account
3. Get your authtoken from dashboard
4. Enter it below:
2z4dv70d99S6fD3J4dkvZreO2up_2SC35TqZjhDUgzc7hhfPm··········
NAME:
   authtoken - save authtoken to configuration file

USAGE:
   ngrok authtoken [command options] [arguments...]

DESCRIPTION:
   The authtoken command modifies your configuration file to include
   the specified authtoken. By default, this configuration file is located
   at $HOME/.ngrok2/ngrok.yml

   The ngrok.com service requires that you sign up for an account to use
   many advanced service features. In order to associate your client with
   an account, it must pass a secret token to the ngrok.com service when it
   starts up. Instead of passing this authtoken on every invocation, you may
   use this command to save it into your configuration file so that your
   client always authenticates you properly.

EXAMPLE:
    ngrok authtoken BDZIXnhJt2HNWLXyQ5PM_qCaBq0W2sNFcCa0rfTZd

OPTIONS:
   --co

In [29]:
# ========================================
# CELL 4: RUN STREAMLIT APP
# ========================================

import subprocess
import threading
import time
import requests

def run_streamlit():
    """Run Streamlit app"""
    subprocess.run(["streamlit", "run", "disaster_dashboard_app.py", "--server.port", "8502", "--server.address", "0.0.0.0"])


def run_ngrok():
    """Run ngrok tunnel"""
    time.sleep(3)  # Wait for Streamlit to start
    subprocess.run(["./ngrok", "http", "8502"])

print("🚀 Starting Streamlit Dashboard...")
print("=" * 50)

# Start Streamlit in background
streamlit_thread = threading.Thread(target=run_streamlit)
streamlit_thread.daemon = True
streamlit_thread.start()

print("✅ Streamlit app started!")
print("📡 Starting ngrok tunnel...")
print("⚠️ Make sure you've set up ngrok authentication first!")

# Uncomment the line below to start ngrok
run_ngrok()

print("🌐 Your Streamlit app will be available at the ngrok URL!")
print("📋 Features available in the dashboard:")
print("   - 🏠 Overview: Key metrics and charts")
print("   - 📊 Data Analysis: Detailed visualizations")
print("   - 🔮 AI Prediction: Real-time disaster prediction")
print("   - 📈 Risk Assessment: Risk scoring and analysis")
print("   - 💾 Data Upload: Dataset management")

🚀 Starting Streamlit Dashboard...
✅ Streamlit app started!
📡 Starting ngrok tunnel...
⚠️ Make sure you've set up ngrok authentication first!
🌐 Your Streamlit app will be available at the ngrok URL!
📋 Features available in the dashboard:
   - 🏠 Overview: Key metrics and charts
   - 📊 Data Analysis: Detailed visualizations
   - 🔮 AI Prediction: Real-time disaster prediction
   - 📈 Risk Assessment: Risk scoring and analysis
   - 💾 Data Upload: Dataset management


In [34]:
# ========================================
# ALTERNATIVE: LOCAL TESTING (For Colab)
# ========================================

# If you want to test locally in Colab without ngrok:
print("\n" + "="*50)
print("🔧 ALTERNATIVE: Local Testing")
print("="*50)
print("Run this command in a new cell to start locally:")
print("!streamlit run disaster_dashboard_app.py --server.port 8502")
print("\nThen use Colab's built-in port forwarding:")
print("Click the 'Open in new tab' button that appears")


🔧 ALTERNATIVE: Local Testing
Run this command in a new cell to start locally:
!streamlit run disaster_dashboard_app.py --server.port 8502

Then use Colab's built-in port forwarding:
Click the 'Open in new tab' button that appears


In [41]:
from pyngrok import ngrok

# Ganti dengan authtoken Anda yang disalin dari ngrok dashboard
ngrok.set_auth_token('2z4dv70d99S6fD3J4dkvZreO2up_2SC35TqZjhDUgzc7hhfPm')

# Membuka ngrok tunnel untuk port 8502 (atau port lain yang Anda pilih)
public_url = ngrok.connect(8502)
print(f"Streamlit app will be available at {public_url}")


Streamlit app will be available at NgrokTunnel: "https://5a9f-34-73-27-220.ngrok-free.app" -> "http://localhost:8502"
